# Umwalimu AI — New Student Guidance Intent Dataset

This notebook replaces the former lesson-tutoring intent experiment with the current project scope: a static-response **New Student Guide** for Streamlit.

The deployed chatbot currently uses deterministic phrase routing and fixed response templates. This notebook audits the new labelled dataset and trains/evaluates a TF-IDF + Logistic Regression intent-classification baseline for experimentation. The deployed app does not depend on the generated model file, so it remains usable without model downloads or API keys.

The examples are curated demonstration examples, not real student interviews or verified institutional policy data.

## 1. Imports and reproducible setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib

DATA_PATH = Path('dataset/intents.csv') if Path('dataset/intents.csv').exists() else Path('intents.csv')
OUTPUT_PATH = DATA_PATH.parent / 'intent_classifier.joblib'
RANDOM_STATE = 42
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Dataset quality checks

In [ ]:
required = {'text', 'intent', 'source', 'split'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
assert df[list(required)].notna().all().all(), 'Missing required values found'
assert (df['text'].str.strip().str.len() > 0).all(), 'Empty messages found'
print('Rows:', len(df))
print('Intents:', sorted(df.intent.unique()))
print('Duplicates:', df.text.str.casefold().str.strip().duplicated().sum())
display(df.groupby(['intent', 'split']).size().unstack(fill_value=0))

## 3. Visualise intent distribution

In [ ]:
df['intent'].value_counts().sort_values().plot(kind='barh', title='New-student guidance examples by intent')
plt.xlabel('Number of examples')
plt.tight_layout()
plt.show()

## 4. Train and evaluate a baseline classifier

In [ ]:
train_df = df[df.split == 'train']
test_df = df[df.split == 'test']
X_train, y_train = train_df.text, train_df.intent
X_test, y_test = test_df.text, test_df.intent

model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), strip_accents='unicode')),
    ('classifier', LogisticRegression(max_iter=1500, class_weight='balanced', random_state=RANDOM_STATE))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, predictions), 3))
print(classification_report(y_test, predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, xticks_rotation=45)
plt.tight_layout()
plt.show()

## 5. Save the experimental model

In [ ]:
joblib.dump(model, OUTPUT_PATH)
print('Saved:', OUTPUT_PATH)
loaded_model = joblib.load(OUTPUT_PATH)
sample = 'I am new. Where should I go first?'
print(sample, '=>', loaded_model.predict([sample])[0])

## 6. Limitations and next steps

- The dataset contains curated demonstration phrases, not a representative sample of actual students.
- Eight examples per intent are too few for a reliable production classifier; the held-out split is only a basic smoke test.
- Phrase overlap and language variation can cause misclassification. Collect consent-based, anonymised real student questions before claiming performance.
- The deployed Streamlit application deliberately uses transparent rules and fixed responses for predictable behaviour. It does not use this model file.
- The application has no verified school-specific directory. Never invent office locations, room numbers, fees, deadlines, dates, or official policies. Add institution-approved data before offering exact answers.
- Safety, bullying, abuse, threats, and urgent medical concerns should be directed to responsible human staff or emergency support.
